# HASTIKA @ ICON-2026 — Kaggle training notebook

**Settings (panel bên phải):** Accelerator = **GPU T4 ×2** · Internet = **On**

Notebook này **không chứa code** — nó clone [`trong5nhan6/Text`](https://github.com/trong5nhan6/Text) rồi gọi các
entrypoint của repo. Sửa code ở máy → `git push` → chạy lại cell dưới đây là có bản mới,
**không cần upload lại notebook**.

Quy trình: 1) repo → 2) data → 3) TF-IDF → 4) transformers → 5) evaluate → 6) submission.
Kết quả nằm trong `/kaggle/working/repo/` và được giữ lại khi **Save Version (Save & Run All)**.

In [ ]:
import os, subprocess, sys

REPO, BRANCH, WORK = "trong5nhan6/Text", "main", "/kaggle/working"

# Repo Public -> clone ẩn danh, không cần gì thêm.
# Nếu sau này đổi sang Private: Add-ons ▸ Secrets ▸ New secret, tên GH_TOKEN,
# giá trị = GitHub Personal Access Token (scope "repo"). Cell này tự dò.
TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
    print("dùng GH_TOKEN từ Kaggle Secrets")
except Exception:
    pass

url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
hide = (lambda s: s.replace(TOKEN, "***")) if TOKEN else (lambda s: s)   # không in token ra log

os.chdir(WORK)
cmd = (["git", "-C", "repo", "pull", "--ff-only"] if os.path.isdir("repo/.git")
       else ["git", "clone", "--depth", "1", "-b", BRANCH, url, "repo"])
r = subprocess.run(cmd, capture_output=True, text=True)
print(hide((r.stdout + r.stderr).strip()))
if r.returncode:
    raise SystemExit("git thất bại — kiểm tra Internet = On, repo Public (hoặc secret GH_TOKEN)")

os.chdir(f"{WORK}/repo"); sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
!pip -q install ftfy sentencepiece tiktoken
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| cuda", torch.cuda.is_available())
# ModernBERT cần transformers >= 4.48:  !pip -q install -U "transformers>=4.48"

## 1) Dữ liệu

CSV của ban tổ chức nằm sẵn trong repo (`data/raw/`), nên chỉ cần làm sạch + chia split.
Một lát **90% fit / 10% chấm điểm** (`data.val_ratio`), phân tầng theo nhãn và cố định bởi
`data.split_seed` — mọi model dùng chung một lát nên blend được với nhau.

In [ ]:
!ls data/raw
!python -m src.data.preprocessing

**Khi test phát hành (20/9)** — chọn 1 trong 2 cách rồi chạy lại `preprocessing`:

In [ ]:
# Cách 1 — test đã được push vào repo: chỉ cần chạy lại cell git pull ở trên.
# Cách 2 — upload test thành Kaggle Dataset rồi copy vào data/raw:
# !cp /kaggle/input/<ten-dataset>/*test*.csv data/raw/
# !python -m src.data.preprocessing

## 2) B0 — TF-IDF (CPU, ~30 giây mỗi run)

`C` là tham số điều chuẩn của mô hình tuyến tính, theo nghĩa **nghịch đảo**: `C` nhỏ = phạt trọng số
mạnh = model đơn giản (dễ underfit); `C` lớn = ưu tiên khớp dữ liệu = dễ overfit. Ở đây đặc trưng
TF-IDF có tới 300k chiều trên vài nghìn dòng train, nên `C` là siêu tham số quan trọng nhất của B0.

`train.py` fit một model cho **mỗi giá trị trong `model.C_grid`** rồi giữ cái có macro-F1 cao nhất
trên lát held-out. Giá trị được chọn hiện trong `results/metrics.csv`, ví dụ `tfidf-lr C=2`.

LR và SVM **không cùng thang `C`** (loss khác nhau), nên SVM dùng dải nhỏ hơn. Chạy cả hai vì blend
của chúng thường tốt hơn từng cái một.

In [ ]:
!python train.py --config configs/tfidf.yaml --task a
!python train.py --config configs/tfidf.yaml --task b

In [ ]:
# LinearSVC — run tự đặt tên tfidf_svm, dải C nhỏ hơn LR
!python train.py --config configs/tfidf.yaml --task a --set model.clf=svm "model.C_grid=[0.05,0.1,0.25,0.5,1]"
!python train.py --config configs/tfidf.yaml --task b --set model.clf=svm "model.C_grid=[0.05,0.1,0.25,0.5,1]"

In [ ]:
# Dò C mịn hơn quanh giá trị vừa chọn (nhớ đổi --run_name, nếu không script từ chối chạy):
# !python train.py --config configs/tfidf.yaml --task b --run_name tfidf_lr_fine --set "model.C_grid=[0.25,0.5,1,1.5,2,3,4]"

## 3) B1 — Transformers

### Config đang dùng

`configs/base.yaml` là mặc định chung; mỗi model chỉ ghi đè phần khác biệt qua `_base_: base.yaml`.
Cell dưới in ra **config đã gộp** — đúng những giá trị `train.py` sẽ chạy.

In [ ]:
import yaml
from src.utils.config import load_config, run_name

SHOW = 'muril'          # đổi để xem config khác: tfidf muril roberta indicbert bert deberta modernbert
SHOW_TASK = 'b'

print(f"--- configs/{SHOW}.yaml (phan ghi de) ".ljust(72, "-"))
print(open(f'configs/{SHOW}.yaml', encoding='utf-8').read())
cfg = load_config(f'configs/{SHOW}.yaml', task=SHOW_TASK)
print(f"--- config da gop, task={SHOW_TASK}, run se ten la '{run_name(cfg)}' ".ljust(72, "-"))
print(yaml.safe_dump({k: cfg[k] for k in ('seed', 'data', 'model', 'training', 'checkpoint')},
                     sort_keys=False, allow_unicode=True))

### Chỉnh siêu tham số ngay ở đây

**Giá trị đang thấy trong `OVERRIDES` chính là mặc định trong `configs/base.yaml`** (sinh tự động
lúc build notebook), nên bỏ dấu `#` mà không sửa gì thì **không đổi gì cả**. Sửa giá trị rồi mới có
tác dụng — cell tự so với `base.yaml` lúc chạy và chỉ báo những khoá thật sự khác.

Nhớ đặt `RUN_SUFFIX` khi bạn đổi siêu tham số: run cũ và run mới sẽ có tên khác nhau nên không đè
lên nhau và so sánh được với nhau. Nếu để trống mà siêu tham số đã đổi, `train.py` sẽ **từ chối chạy**
thay vì âm thầm trộn kết quả.

In [ ]:
OVERRIDES = {          # gia tri = mac dinh trong configs/base.yaml, sua roi moi co tac dung
    # --- huan luyen ---
    # 'training.epochs':                    30,         # tran, khong phai muc tieu; cung la do dai lich LR
    # 'training.early_stopping_patience':   5,          # dung khi macro-F1 khong cai thien bay nhieu epoch lien
    # 'training.lr':                        2e-05,      # learning rate cua backbone (head dung head_lr)
    # 'training.batch_size':                64,
    # 'training.grad_accum':                1,          # tang len khi giam batch_size, de giu batch hieu dung
    # 'training.loss':                      'auto',     # auto | ce | wce | focal   (auto: task a -> ce, task b -> wce)
    # 'training.label_smoothing':           0.0,        # bi bo qua khi loss=focal
    # --- du lieu ---
    # 'data.max_len':                       96,         # Religion / Geo-political dai hon, hay bi cat o 96
    # --- model ---
    # 'model.pooling':                      'cls',      # cls | mean
    # 'model.dropout':                      0.1,
    # 'seed':                               42,
    # --- dia ---
    # 'checkpoint.save':                    'best',     # best | none   (none tiet kiem ~0.5 GB moi run)
}
RUN_SUFFIX = ''        # vi du '_e8' -> run ten muril_wce_s42_e8. BAT BUOC khi doi gia tri that su.

# doi data.val_ratio / data.split_seed se chia lai split, moi ket qua cu se het so sanh duoc

import yaml
_base = yaml.safe_load(open('configs/base.yaml', encoding='utf-8'))
def _default_of(k):
    node = _base
    for part in k.split('.'):
        node = node[part]
    return node

changed = {k: v for k, v in OVERRIDES.items() if _default_of(k) != v}
ARGS = " ".join(f"{k}={v}" for k, v in OVERRIDES.items())
ARGS = (f"--set {ARGS}" if ARGS else "") + (f" --run_suffix {RUN_SUFFIX}" if RUN_SUFFIX else "")

print("them vao lenh train:", ARGS or "(khong co, dung mac dinh)")
if changed:
    for k, v in changed.items():
        print(f"  doi that su: {k}  {_default_of(k)} -> {v}")
    if not RUN_SUFFIX:
        print("!! RUN_SUFFIX dang trong -> train.py se tu choi chay de khong de len run cu")
elif OVERRIDES:
    print("  (cac dong da bo # deu dang giu nguyen mac dinh -> khong doi gi)")

### Train

Cấu hình hiện tại: `epochs: 30` là **trần**, `early_stopping_patience: 5` mới là thứ quyết định
lúc dừng — train tiếp cho tới khi macro-F1 không cải thiện suốt 5 epoch liền.

Trên T4, MuRIL/XLM-R base với `batch_size: 64` mất ~40–60 giây mỗi epoch (task A ~90 step,
task B ~45 step). Thực tế thường dừng quanh epoch 8–15, tức **~8–15 phút mỗi run**, 4 run
≈ 35–60 phút. Nếu sắp hết giờ session thì giảm `CONFIGS` hoặc `TASKS` lại.

- Run đã xong → chạy lại sẽ bỏ qua, không train lại.
- Mỗi run lưu 1 checkpoint fp16 (~0.5 GB với model base); `/kaggle/working` giới hạn ~20 GB.

> **`epochs` vừa là trần vừa là độ dài lịch learning rate.** `trainer.py` tính
> `steps = step_mỗi_epoch × epochs`, warmup = 10% số đó, rồi LR giảm tuyến tính về 0 ở step cuối.
> Vì `epochs: 30` lớn hơn nhiều so với lúc thực sự dừng, warmup kéo dài 3 epoch đầu và LR **gần như
> không giảm** trong suốt quá trình train (epoch 6 vẫn còn ~89% đỉnh, epoch 15 còn ~56%). Đây là
> đánh đổi đã biết: model chạy ở LR cao ổn định thay vì có giai đoạn anneal cuối. Muốn có anneal thì
> đặt `training.epochs` sát số epoch kỳ vọng — sửa được ngay ở cell OVERRIDES phía trên.

In [ ]:
import time

CONFIGS = ['muril', 'roberta']        # thêm: 'indicbert', 'bert', 'deberta', 'modernbert'
TASKS   = ['a', 'b']

t0 = time.time()
for i, c in enumerate(CONFIGS):
    for j, t in enumerate(TASKS):
        n = i * len(TASKS) + j + 1
        print("")
        print("=" * 72)
        print(f"[{n}/{len(CONFIGS) * len(TASKS)}]  config = {c}   |   task = {t}   "
              f"|   {time.strftime('%H:%M:%S')}   |   +{(time.time() - t0) / 60:.1f} phut")
        print("=" * 72, flush=True)
        !python train.py --config configs/{c}.yaml --task {t} {ARGS}
print("")
print(f"xong {len(CONFIGS) * len(TASKS)} run trong {(time.time() - t0) / 60:.1f} phut")

In [ ]:
# Ví dụ biến thể:
# !python train.py --config configs/muril.yaml --task b --set training.loss=focal
# !python train.py --config configs/muril.yaml --task b --set data.max_len=128 --run_name muril_len128
# !python train.py --config configs/roberta.yaml --task b --run_name xlmr_large --set model.name=xlm-roberta-large training.lr=1e-5 training.batch_size=16 training.grad_accum=2

In [ ]:
!du -sh checkpoints/*/* 2>/dev/null; df -h /kaggle/working | tail -1
# xoá run không cần:  !rm -rf checkpoints/b/<run_name> results/b/<run_name>

## 4) Evaluate

Mọi run đều chấm trên cùng một lát held-out (`n_eval` dòng) nên so sánh và blend được trực tiếp.

In [ ]:
import pandas as pd
display(pd.read_csv('results/metrics.csv'))
!python evaluate.py --task a
!python evaluate.py --task b

In [ ]:
# blend + tối ưu trọng số trên lát eval (đổi tên run theo bảng trên)
!python evaluate.py --task a --runs tfidf_lr tfidf_svm muril_ce_s42 roberta_ce_s42 --optimize
!python evaluate.py --task b --runs tfidf_lr tfidf_svm muril_wce_s42 roberta_wce_s42 --optimize

In [ ]:
from IPython.display import Image, display
for t in ('a', 'b'):
    p = f'results/{t}/_blend/confusion.png'
    if os.path.exists(p):
        print(p); display(Image(p))

## 5) Submission

### Train nhiều model cùng lúc thì file nằm đâu?

**Mỗi run một thư mục riêng, không cái nào đè cái nào.** Tên run mặc định là
`<config>_<loss>_s<seed>`, task là thư mục cha:

```
results/
├── a/                              Task A
│   ├── tfidf_lr/                   eval.npy  val.npy  [test.npy]  metrics.json  config.yaml
│   ├── tfidf_svm/
│   ├── muril_ce_s42/               ← configs/muril.yaml  --task a
│   └── roberta_ce_s42/             ← configs/roberta.yaml --task a
├── b/                              Task B  (loss mặc định là wce nên tên là _wce_)
│   ├── tfidf_lr/  tfidf_svm/  muril_wce_s42/  roberta_wce_s42/
│   └── _blend/                     kết quả blend gần nhất
└── metrics.csv                     1 dòng cho mỗi run, cả 2 task
checkpoints/{task}/{run}/           trọng số fp16 của run đó
```

### Ba tập dữ liệu, đừng nhầm

| File trong run | Là gì | Có nhãn? | Dùng để |
|---|---|---|---|
| `eval.npy` | lát held-out 10% **cắt ra từ train** | ✅ | chấm điểm nội bộ, chọn model, tìm trọng số blend |
| `val.npy` | `*_validation_inputs.csv` **của BTC** | ❌ | **nộp phase Development** |
| `test.npy` | `*_test_inputs.csv` (phát 20/9) | ❌ | **nộp phase Evaluation** |

Chữ "val" xuất hiện ở hai nghĩa khác nhau: lát held-out (có nhãn, để bạn tự chấm) và file
validation của BTC (không nhãn, để nộp). `eval.npy` là cái đầu, `val.npy` là cái sau.

### Nộp thế nào

Codabench có **2 leaderboard riêng biệt**, mỗi task nộp **một** `predictions.csv`. Bạn chỉ có
**20 lượt nộp**, nên đừng nộp từng model — chọn theo điểm held-out ở mục 4 rồi nộp bản tốt nhất.

- Run train *sau* khi có test → `--split test`
- Run train *trước* khi có test → mode 2 (`--checkpoints ... --input ...`), không cần train lại

**Cách 1 — mỗi run một file nộp riêng** (để đối chiếu, đặt tên theo task + run):

In [ ]:
import glob, os
for t in ('a', 'b'):
    for f in sorted(glob.glob(f'results/{t}/*/val.npy')):
        run = os.path.basename(os.path.dirname(f))
        !python inference.py --task {t} --runs {run} --split val --tag {run}

**Cách 2 — ensemble** (thường tốt hơn; đổi tên run và trọng số theo bảng ở mục 4):

In [ ]:
!python inference.py --task a --runs tfidf_lr tfidf_svm muril_ce_s42 roberta_ce_s42 --split val --tag ens3
!python inference.py --task b --runs tfidf_lr tfidf_svm muril_wce_s42 roberta_wce_s42 --split val --tag ens3
# test, từ checkpoint:
# !python inference.py --task b --checkpoints checkpoints/b/muril_wce_s42 --input data/raw/multiclass_test_inputs.csv --tag ens1

In [ ]:
import glob, shutil
out = '/kaggle/working/submissions'
os.makedirs(out, exist_ok=True)
for z in glob.glob('results/submissions/*/submission.zip'):
    shutil.copy(z, f"{out}/{os.path.basename(os.path.dirname(z))}.zip")
!ls -la /kaggle/working/submissions